In [3]:
import torch
import torch.nn.functional as F
import torch.nn as nn

In [5]:
# 基础版本的MOE


# 首先实现expert
class BasicExerpt(nn.Module):
    def __init__(self,feature_in,feature_out):
        super().__init__()
        self.linear = nn.Linear(feature_in,feature_out)
    
    def forward(self,x):
        x = self.linear(x)
        return x

#然后实现BasicMOE
class BasicMOE(nn.Module):
    def __init__(self,feature_in,feature_out,number_experts):
        super().__init__()
        self.experts = nn.ModuleList(
            [BasicExerpt(feature_in,feature_out) for _ in range(number_experts)]
        )
        self.gate = nn.Linear(feature_in,number_experts)
    def forward(self,x):
        # x : (B,feature_in)
        expert_weight = self.gate(x) #(Batch,number_experts)
        expert_out_list = [
            expert(x).unsqueeze(1) for expert in self.experts  #expert(x):(Batch,feature_out) / expert(x).unsqueeze(1) :(Batch,1,feature_out)
        ]
        
        expert_out = torch.concat(expert_out_list,dim=1) #(Batch,number_experts,feature_out)
        expert_weight = expert_weight.unsqueeze(1) # (Batch,number_experts) -> (Batch,1,number_experts)
        
        out = expert_weight @ expert_out  # (Batch,1,feature_out)
        
        return out.squeeze()
        


In [7]:
x = torch.rand(2,4)
basic_moe = BasicMOE(4,3,2)
out = basic_moe(x)
print(out)

tensor([[ 0.4091, -0.2107, -0.2127],
        [ 0.6676, -0.2067, -0.2660]], grad_fn=<SqueezeBackward0>)


In [16]:
# sparse MOE
#MOE 选择 topK 个专家，然后对这 topK 个专家的输出进行加权求和，并且把输入样本变成了大模型中真实的输入 Shape，(batch, seq_len, hidden_dim)

class MOERouter(nn.Module):
    def __init__(self,hidden_dim,expert_number,top_k):
        super().__init__()
        self.gate = nn.Linear(hidden_dim,expert_number)
        self.expert_number = expert_number
        self.top_k = top_k
    
    def forward(self,hidden_states):
        router_logits = self.gate(hidden_states) # (batch_size * seq_len,exper_number) "每个 token 都要分配一个专家"，所以我们把所有 token 拍扁为一个 batch，统一送进 gate 网络，逐 token 路由。所以这边的输入的hidden_states 的形状是 (B*S,hidden_dim)
        
        #计算路由概率
        routing_probs = F.softmax(router_logits,dim=-1,dtype=torch.float)
        
        #计算topK专家输出
        router_weights,selected_experts = torch.topk(routing_probs,self.top_k,dim=-1) # ( B * S ,top_k)/ router_weights 是 softmax 后的概率中的 top-k /selected_experts是这 top-k 值在原始专家列表中的索引。
        
        
        #挑选出得top K 个专家的权重进行归一化
        router_weights = router_weights / router_weights.sum(dim=-1,keepdim=True)
        router_weights = router_weights.to(hidden_states.dtype)
        
        # 专家掩码 把所有token按照专家纬度打包
        expert_mask = F.one_hot(selected_experts,num_classes=self.expert_number) # (B*S , topk, expert_number)
        expert_mask = expert_mask.permute(2,1,0) # (expert_number, topk, B*S) （哪个专家（第几号专家），这个专家是该 token 的第几个选择（Top-1、Top-2...），总 token 数（batch_size × seq_len）中第几个 token）
        
        return router_logits, router_weights, selected_experts, expert_mask


class MOEConfig:
    def __init__(
            self, 
            hidden_dim, 
            expert_number, 
            top_k, 
            shared_experts_number=2,
        ):
        self.hidden_dim = hidden_dim
        self.expert_number = expert_number
        self.top_k = top_k
        self.shared_experts_number = shared_experts_number
        
        

class SparseMOE(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.hidden_dim = config.hidden_dim

        self.expert_number = config.expert_number
        self.top_k = config.top_k
        
        self.experts = nn.ModuleList(
            [BasicExerpt(self.hidden_dim,self.hidden_dim) for _ in range(config.expert_number)]
        )
        
        self.router = MOERouter(self.hidden_dim,self.expert_number,self.top_k)
    
    def forward(self,x):
        batch_size, seq_len, hidden_dim = x.size()
        
        hidden_states = x.view(-1,hidden_dim) # (B*S , hidden_dim) 不是sample维度而是token维度了
        router_logits , router_weights, selected_experts, expert_mask = self.router(hidden_states)
        
        # 其中 selected_experts_indices shape 是 (b * s, top_k)
        # 其中 expert_mask shape 是 (expert_number, top_k, b * s)
        
        final_hidden_states = torch.zeros((batch_size*seq_len,hidden_dim),dtype=hidden_states.dtype,device=hidden_states.device)
        
        for expert_idx in range(self.expert_number):
            expert_layer = self.experts[expert_idx]
            
            idx, top_x = torch.where(expert_mask[expert_idx])
            # idx 和 top_x 都是一维 tensor
            # idx 的值是 0 或 1, 表示这个 token 是作为当前专家的 top1 还是 top2
            # top_x 的值是 token 在 batch*seq_len 中的位置索引
            # 例如对于 batch_size=2, seq_len=4 的输入:
            # top_x 的值范围是 0-7, 表示在展平后的 8 个 token 中的位置
            # idx 的值是 0/1, 表示这个 token 把当前专家作为其 top1/top2 专家
            current_state = hidden_states[top_x,:] # （selected_token_number, hidden_dim）挑选出topx 位置的token
            
            # router_weights的形状是(b*s , topk)
            current_hidden_states = expert_layer(current_state) * router_weights[top_x, idx].unsqueeze(-1) #(selected_token_number,hidden_dim) * (selected_token_number，1)
            # 把当前专家的输出加到 final_hidden_states 中
            # 方式1 的写法性能更好，并且方式1容易出现
            final_hidden_states.index_add_(0, top_x, current_hidden_states.to(hidden_states.dtype)) 
            
        # 把 final_hidden_states 还原到原来的 shape
        final_hidden_states = final_hidden_states.reshape(batch_size, seq_len, hidden_dim)
            
        return final_hidden_states,router_logits
        
        

def test_token_level_moe():
    x = torch.rand(2, 4, 16)
    config = MOEConfig(16, 2, 2)
    token_level_moe = SparseMOE(config)
    out = token_level_moe(x)
    print(out[0].shape, out[1].shape)

test_token_level_moe()


torch.Size([2, 4, 16]) torch.Size([8, 2])


In [15]:
class ShareExpertMOE(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.moe_model = SparseMOE(config)
        self.shared_experts = nn.ModuleList(
            [BasicExerpt(config.hidden_dim,config.hidden_dim) for _ in range(config.expert_number)]
        )
        
        
    def forward(self,x):
        sparse_model_out, router_logits = self.moe_model(x)
        
        shared_experts_out = [expert(x) for expert in self.shared_experts]
        shared_experts_out = torch.stack(shared_experts_out,dim=0).sum(dim=0)
        
        return shared_experts_out,router_logits



def test_share_expert_moe():
    x = torch.rand(2, 4, 16)
    config = MOEConfig(16, 2, 2)
    share_expert_moe = ShareExpertMOE(config)
    out = share_expert_moe(x)
    print(out[0].shape, out[1].shape)


test_share_expert_moe()

torch.Size([2, 4, 16]) torch.Size([8, 2])


In [ ]:
#模型训练
##训练router 负载均衡 防止只使用部分专家
def switch_load_balance_loss(router_logits,expert_numbers): #router_logits shape (B*S,exper_numbers)
    
    #计算路由概率
    router_probs = F.softmax(router_logits,dim=-1) #(B*S,expert_numbers)
    # 获取每个token的最优专家
    _,selected_experts = torch.topk(router_probs,k=2,dim=-1)
    
    # 创建one-hot矩阵表示选中的专家
    mask = torch.nn.functional.one_hot(selected_experts,expert_numbers).float() #(b*s,top_k,expert_numbers)
    
    # 计算每个专家的期望负载 (理想情况下应该是 1/num_experts)
    expected_load = torch.ones_like(router_probs) / expert_numbers
    
    #计算实际负载
    #每个专家处理的token总数/总token的数量
    
    actual_load = mask.mean(dim=0) #(top_k,experts_nums)
    
     # 计算auxiliary loss
    # 这会惩罚负载分布与期望负载的差异
    aux_loss = torch.sum(actual_load * router_probs.mean(dim=0)) * num_experts   
    ##router_probs.mean(dim=0) shape:(expert_numbers,) dim=0 是按照token维度也就是列
    #actual_load * router_probs.mean(dim=0) (topK,experts_numer)
    #torch.sum(actual_load * router_probs.mean(dim=0)) -> 标量
    
    #应为期望负载和actual_load 只有都接近1/N 才能损失最小
    z_loss = torch.mean(torch.square(router_logits))
    z_loss_weight = 0.001  # 可调整的超参数
    
    # 总损失
    total_loss = aux_loss + z_loss * z_loss_weight
    
    return total_loss

def test_moe_training():
    # Create a simple dataset
    batch_size = 32
    seq_len = 16
    hidden_dim = 32
    num_batches = 100
    
    # Initialize model and optimizer
    config = MOEConfig(hidden_dim=hidden_dim, 
                      expert_number=4,
                      top_k=2,
                      shared_experts_number=2)
    model = ShareExpertMOE(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Training loop
    model.train()
    for batch in range(num_batches):
        # Generate random input data
        x = torch.randn(batch_size, seq_len, hidden_dim)
        target = torch.randn(batch_size, seq_len, hidden_dim)
        
        # Forward pass
        output, router_logits = model(x)

        # Compute losses
        # MSE loss for prediction
        mse_loss = F.mse_loss(output, target)
        
        aux_loss = switch_load_balance_loss(router_logits, config.expert_number)
        # Combined loss
        total_loss = mse_loss + 0.01 * aux_loss
        
        # Backward pass and optimize
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        if batch % 10 == 0:
            print(f"Batch {batch}, Loss: {total_loss.item():.4f} "
                  f"(MSE: {mse_loss.item():.4f}, Aux: {aux_loss.item():.4f})")

# Run the training test
test_moe_training()

In [21]:
import torch
import torch.nn.functional as F

# 参数设定
b, s = 2, 4  # batch size 和 sequence length
num_experts = 3
top_k = 2

# 模拟 router logits（可看作模型输出）
router_logits = torch.randn(b * s, num_experts)

# softmax 得到每个 token 的专家概率分布
router_probs = F.softmax(router_logits, dim=-1)

# 选择 top-k 的专家
_, selected_experts = torch.topk(router_probs, k=top_k, dim=-1)  # shape: (b*s, k)

# 打印一下看看
print("Selected experts:\n", selected_experts)
print("Shape:", selected_experts.shape)

# one-hot 编码：每个 token 的 top-k 分配（变成 multi-hot）
mask = F.one_hot(selected_experts, num_classes=num_experts).float()  # shape: (b*s, k, num_experts)

print("\nMask shape:", mask.shape)
print("Mask example (for token 0 and token 1):")
print(mask)
print(mask[0])  # 第一个 token 的 top-k mask
print(mask[1])  # 第二个 token 的 top-k mask

# 统计实际负载（mean over token维度）
actual_load = mask.mean(dim=0)  # shape: (k, num_experts)

print("\nActual load (top-k expert usage frequency):")
print(actual_load)
print("Shape:", actual_load.shape)


Selected experts:
 tensor([[1, 2],
        [1, 0],
        [0, 2],
        [0, 1],
        [1, 2],
        [0, 2],
        [0, 1],
        [2, 1]])
Shape: torch.Size([8, 2])

Mask shape: torch.Size([8, 2, 3])
Mask example (for token 0 and token 1):
tensor([[[0., 1., 0.],
         [0., 0., 1.]],

        [[0., 1., 0.],
         [1., 0., 0.]],

        [[1., 0., 0.],
         [0., 0., 1.]],

        [[1., 0., 0.],
         [0., 1., 0.]],

        [[0., 1., 0.],
         [0., 0., 1.]],

        [[1., 0., 0.],
         [0., 0., 1.]],

        [[1., 0., 0.],
         [0., 1., 0.]],

        [[0., 0., 1.],
         [0., 1., 0.]]])
tensor([[0., 1., 0.],
        [0., 0., 1.]])
tensor([[0., 1., 0.],
        [1., 0., 0.]])

Actual load (top-k expert usage frequency):
tensor([[0.5000, 0.3750, 0.1250],
        [0.1250, 0.3750, 0.5000]])
Shape: torch.Size([2, 3])


In [ ]:
##如果有多层moe 需要计算辅助损失之合
class ShareExpertMOE(nn.Module):
    def __init__(self, config, num_layers=3):
        super().__init__()
        self.moe_layers = nn.ModuleList(
            [SparseMOE(config) for _ in range(num_layers)]
        )
        self.shared_experts = nn.ModuleList(
            [BasicExpert(config.hidden_dim, config.hidden_dim) for _ in range(config.shared_experts_number)]
        )

    def forward(self, x):
        aux_losses = []
        for moe_layer in self.moe_layers:
            x, router_logits = moe_layer(x)
            aux_losses.append(router_logits)

        # shared experts
        shared_out = sum(expert(x) for expert in self.shared_experts)
        return x + shared_out, aux_losses  # 返回所有层的 router_logits
